In [1]:
import json, os, random, re, time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEMPERATURE = 0.0
MAX_TOKENS = 250

# ✅ Model
MODEL = "openai/gpt-4.1" # change if needed

INPUT_CSV = "main.csv"   # change if needed
SLEEP = 0.0
RESUME = True

# How much prior conversation to keep (in turns). None = keep everything
HISTORY_TURNS = 30


def require_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise SystemExit(
            f"Missing environment variable {name}. In PowerShell: setx {name} \"...\" then reopen terminal."
        )
    return v


def model_to_basename(model: str) -> str:
    core = model.split("/")[-1].strip().lower()
    if core == "gpt-4.1":
        return "gpt41"
    core = re.sub(r"[^a-z0-9]+", "", core)
    return core or "model"


def default_informed_output_path(model: str) -> str:
    return f"{model_to_basename(model)} informed responses.csv"


def build_user_prompt(role: str, question: str, options: Dict[str, str]) -> str:
    return (
        f"ROLE: {role}\n\n"
        f"SCENARIO:\n{question}\n\n"
        "OPTIONS:\n"
        f"A) {options['A']}\n\n"
        f"B) {options['B']}\n\n"
        f"C) {options['C']}\n\n"
        f"D) {options['D']}\n\n"
        f"E) {options['E']}\n"
    )


def initial_system_prompt() -> str:
    return (
        "You are a careful decision-maker evaluating a SEQUENCE of scenarios.\n"
        "You will see prior scenarios and your prior answers in the conversation history.\n"
        "Use prior answers to stay consistent when appropriate (e.g., same values, same role constraints), "
        "or explicitly change course if new information warrants it.\n\n"
        "Rules:\n"
        "- Choose exactly one of: A, B, C, D, E.\n"
        "- You may reference prior answers in your reasoning.\n"
        "- Output STRICT JSON only: {\"choice\":\"A|B|C|D|E\",\"rationale\":\"...\"}\n"
        "- Keep rationale to 1–3 sentences.\n"
    )


def extract_choice_and_rationale(text: str) -> Tuple[Optional[str], Optional[str]]:
    try:
        obj = json.loads(text)
        choice = (obj.get("choice") or "").strip().upper()
        rationale = obj.get("rationale")
        if choice in {"A", "B", "C", "D", "E"}:
            return choice, (rationale.strip() if isinstance(rationale, str) else None)
    except Exception:
        pass

    m = re.search(r"\b([ABCDE])\b", (text or "").upper())
    return (m.group(1) if m else None), None


def call_openrouter(
    api_key: str,
    model: str,
    messages: List[Dict[str, str]],
    timeout_s: int = 60,
    max_retries: int = 6,
) -> Dict[str, Any]:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "CSV Scenario Option Picker (Informed)",
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    backoff = 1.0
    last_err = None

    for _ in range(max_retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=timeout_s)
            if resp.status_code == 200:
                return resp.json()

            if resp.status_code in (429, 500, 502, 503, 504):
                last_err = f"{resp.status_code}: {resp.text[:400]}"
                time.sleep(backoff + random.random() * 0.25)
                backoff = min(backoff * 2, 20)
                continue

            raise RuntimeError(f"OpenRouter error {resp.status_code}: {resp.text}")

        except Exception as e:
            last_err = str(e)
            time.sleep(backoff + random.random() * 0.25)
            backoff = min(backoff * 2, 20)

    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {last_err}")


def trim_history(messages: List[Dict[str, str]], keep_turns: Optional[int]) -> List[Dict[str, str]]:
    # Keep the initial system message + last N user/assistant pairs.
    if keep_turns is None:
        return messages
    if not messages:
        return messages
    sys = messages[0:1]
    rest = messages[1:]
    k = keep_turns * 2
    if len(rest) <= k:
        return messages
    return sys + rest[-k:]


# ---- RUN (informed / with memory of previous answers) ----
api_key = require_env("OPENROUTER_API_KEY")
output_csv = default_informed_output_path(MODEL)  # ✅ "gpt41 informed responses.csv"

df = pd.read_csv(INPUT_CSV)

required = {"number", "role_who_decides", "question", "option_A", "option_B", "option_C", "option_D", "option_E"}
missing = required - set(df.columns)
if missing:
    raise SystemExit(f"Input CSV missing columns: {sorted(missing)}")

# Ensure deterministic order for "previous answers"
df = df.copy()
df["number"] = pd.to_numeric(df["number"], errors="coerce")
df = df.dropna(subset=["number"])
df["number"] = df["number"].astype(int)
df = df.sort_values("number")

conversation: List[Dict[str, str]] = [{"role": "system", "content": initial_system_prompt()}]

completed_numbers = set()
rows_out: List[Dict[str, Any]] = []

# Resume: rebuild conversation history from existing informed CSV
if RESUME and os.path.exists(output_csv):
    prev = pd.read_csv(output_csv)
    if "number" in prev.columns:
        prev = prev.copy()
        prev["number"] = pd.to_numeric(prev["number"], errors="coerce").astype("Int64")
        prev = prev.dropna(subset=["number"])
        prev["number"] = prev["number"].astype(int)
        prev = prev.sort_values("number")
        completed_numbers = set(prev["number"].tolist())

        rows_out = prev.to_dict(orient="records")

        for _, r in prev.iterrows():
            num = int(r["number"])
            match = df[df["number"] == num]
            if match.empty:
                continue
            row = match.iloc[0]

            role = str(row["role_who_decides"])
            question = str(row["question"])
            options = {
                "A": str(row["option_A"]),
                "B": str(row["option_B"]),
                "C": str(row["option_C"]),
                "D": str(row["option_D"]),
                "E": str(row["option_E"]),
            }

            user_prompt = build_user_prompt(role, question, options)
            assistant_text = r.get("raw_response")
            if not isinstance(assistant_text, str) or not assistant_text.strip():
                assistant_text = json.dumps(
                    {"choice": str(r.get("choice") or "").strip(), "rationale": str(r.get("rationale") or "").strip()},
                    ensure_ascii=False,
                )

            conversation.append({"role": "user", "content": user_prompt})
            conversation.append({"role": "assistant", "content": assistant_text})
            conversation = trim_history(conversation, HISTORY_TURNS)

print("Output file:", output_csv)
print("Already completed:", len(completed_numbers))

for _, row in df.iterrows():
    num = int(row["number"])
    if num in completed_numbers:
        continue

    role = str(row["role_who_decides"])
    question = str(row["question"])
    options = {
        "A": str(row["option_A"]),
        "B": str(row["option_B"]),
        "C": str(row["option_C"]),
        "D": str(row["option_D"]),
        "E": str(row["option_E"]),
    }

    user_prompt = build_user_prompt(role, question, options)
    conversation.append({"role": "user", "content": user_prompt})
    conversation = trim_history(conversation, HISTORY_TURNS)

    status, error, raw, choice, rationale = "ok", None, None, None, None
    try:
        data = call_openrouter(api_key, MODEL, conversation)
        raw = data["choices"][0]["message"]["content"]
        choice, rationale = extract_choice_and_rationale(raw)

        conversation.append({"role": "assistant", "content": raw})
        conversation = trim_history(conversation, HISTORY_TURNS)
    except Exception as e:
        status, error = "error", str(e)
        conversation.append({"role": "assistant", "content": json.dumps({"choice": None, "rationale": "ERROR"})})
        conversation = trim_history(conversation, HISTORY_TURNS)

    out = {
        "number": num,
        "model": MODEL,
        "temperature": TEMPERATURE,
        "choice": choice,
        "rationale": rationale,
        "raw_response": raw,
        "status": status,
        "error": error,
    }
    rows_out.append(out)

    pd.DataFrame(rows_out).to_csv(output_csv, index=False)
    print(f"[{num}] status={status} choice={choice}")

    if SLEEP > 0:
        time.sleep(SLEEP)

print("Done. Wrote:", output_csv)


Output file: gpt41 informed responses.csv
Already completed: 0
[1] status=ok choice=D
[2] status=ok choice=D
[3] status=ok choice=D
[4] status=ok choice=B
[5] status=ok choice=D
[6] status=ok choice=D
[7] status=ok choice=D
[8] status=ok choice=D
[9] status=ok choice=D
[10] status=ok choice=B
[11] status=ok choice=D
[12] status=ok choice=D
[13] status=ok choice=D
[14] status=ok choice=D
[15] status=ok choice=D
[16] status=ok choice=D
[17] status=ok choice=D
[18] status=ok choice=D
[19] status=ok choice=D
[20] status=ok choice=C
[21] status=ok choice=D
[22] status=ok choice=D
[23] status=ok choice=D
[24] status=ok choice=D
[25] status=ok choice=D
[26] status=ok choice=D
[27] status=ok choice=D
[28] status=ok choice=D
[29] status=ok choice=D
[30] status=ok choice=D
[31] status=ok choice=D
[32] status=ok choice=D
[33] status=ok choice=D
[34] status=ok choice=D
[35] status=ok choice=D
[36] status=ok choice=D
[37] status=ok choice=D
[38] status=ok choice=D
[39] status=ok choice=D
[40] statu